# Keep fse where the text already is
This is a measurement harness, not an HTTP deployment recipe.
It measures GloVe-300 compute, a tiny stdlib localhost HTTP response, and the serialized size of one real vector.
The server returns a fixed vector. Its request time measures local transport and JSON transfer without embedding compute.
A persistent connection avoids a TCP connection per request. Real networks, concurrency and batching change these costs.
Use these measurements to decide whether moving text through a service helps your application.
The default recommendation is to run fse in the process that already has the text, because that avoids transferring text and vectors.

Run on an otherwise quiet machine. This notebook refuses load averages of 5 or more before and after timing.
When sharing a benchmark machine, acquire its benchmark lock and pause competing heavy work before execution.
The repository's notebooks are intentionally not executed in CI.

In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory
import os
import platform
from datetime import date

scratch = TemporaryDirectory(prefix="fse-recipe-")
work = Path(scratch.name)
# Third-party caches also belong to this temporary directory.
for name in ("HF_HOME", "XDG_CACHE_HOME", "NUMBA_CACHE_DIR", "MPLCONFIGDIR"):
    os.environ[name] = str(work / name.lower())
os.environ["ANONYMIZED_TELEMETRY"] = "False"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
import numpy as np
import fse
print(date.today().isoformat(), platform.platform(), platform.machine())

In [ ]:
import csv
import statistics
import time
import json
import socket
import threading
import http.client
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer
from gensim.models import KeyedVectors
from huggingface_hub import hf_hub_download
revision = "4e5ad402664236b85be45a0d107b95259ba8a909"
files = ["glove-wiki-gigaword-300.model", "glove-wiki-gigaword-300.model.vectors.npy"]
paths = []
for filename in files:
    cached = Path.home() / ".cache/huggingface/hub/models--fse--glove-wiki-gigaword-300/snapshots" / revision / filename
    paths.append(cached if cached.is_file() else Path(hf_hub_download("fse/glove-wiki-gigaword-300", filename, revision=revision, cache_dir=work / "hub")))
wv = KeyedVectors.load(str(paths[0]), mmap="r")
# Only the vocabulary used by this fixed workload is needed. This is not a full-model cache benchmark.
import re
repository = Path.cwd().parent if Path.cwd().name == "recipes" else Path.cwd()
with (repository / "evaluation/sts-test.csv").open() as stream:
    sample = [r[5] for r in csv.reader(stream, delimiter="\t", quoting=csv.QUOTE_NONE)][:1000]
words_ = sorted({w for t in sample for w in re.findall(r"[a-z0-9]+", t.lower()) if w in wv})
encoder = fse.Encoder.from_arrays(np.array([wv[w] for w in words_], dtype=np.float32), words=words_)
workload = sample * 10
encoder.encode(workload, workers=1)  # warm tokenizer and native pipeline
vector = encoder.encode([sample[0]], workers=1)[0]
assert vector.shape == (300,)
binary = vector.astype("<f4").tobytes()
json_bytes = json.dumps(vector.tolist(), separators=(",", ":"), allow_nan=False).encode("utf-8")
print({"binary_bytes": len(binary), "JSON_bytes": len(json_bytes), "dimensions": len(vector)})

In [ ]:
class Handler(BaseHTTPRequestHandler):
    protocol_version = "HTTP/1.1"
    def setup(self):
        super().setup()
        self.connection.setsockopt(socket.IPPROTO_TCP, socket.TCP_NODELAY, 1)
    def do_GET(self):
        self.send_response(200)
        self.send_header("Content-Type", "application/json")
        self.send_header("Content-Length", str(len(json_bytes)))
        self.end_headers()
        self.wfile.write(json_bytes)
    def log_message(self, *args):
        pass

load_before = os.getloadavg()[0]
if load_before >= 5:
    raise RuntimeError(f"load {load_before:.2f} is too high for timing")
server = ThreadingHTTPServer(("127.0.0.1", 0), Handler)
thread = threading.Thread(target=server.serve_forever, daemon=True)
thread.start()
connection = http.client.HTTPConnection("127.0.0.1", server.server_port, timeout=10)
try:
    connection.request("GET", "/")
    assert connection.getresponse().read() == json_bytes
    compute, single, requests = [], [], []
    for _ in range(5):
        start = time.perf_counter_ns()
        encoder.encode(workload, workers=1)
        compute.append((time.perf_counter_ns()-start)/len(workload)/1000)
    for _ in range(100):
        start = time.perf_counter_ns()
        encoder.encode([sample[0]], workers=1)
        single.append((time.perf_counter_ns()-start)/1000)
        start = time.perf_counter_ns()
        connection.request("GET", "/")
        assert connection.getresponse().read() == json_bytes
        requests.append((time.perf_counter_ns()-start)/1000)
finally:
    connection.close()
    server.shutdown()
    server.server_close()
    thread.join()
load_after = os.getloadavg()[0]
if load_after >= 5:
    raise RuntimeError(f"load {load_after:.2f} invalidates the timing")
measurements = {
    "date": date.today().isoformat(), "machine": platform.platform(), "processor": platform.machine(),
    "table": "GloVe-300 workload vocabulary only", "vocabulary_rows": len(words_), "workers": 1,
    "sentences_per_batch": len(workload), "batch_repetitions": 5, "single_and_HTTP_repetitions": 100,
    "compute_batch_us_per_sentence_median": statistics.median(compute),
    "compute_single_sentence_us_median": statistics.median(single),
    "HTTP_keepalive_JSON_us_median": statistics.median(requests),
    "load_before": load_before, "load_after": load_after,
    "binary_bytes": len(binary), "JSON_bytes": len(json_bytes),
}
print(json.dumps(measurements, indent=2))

The batch compute figure amortizes one encoder call across 10,000 sentences.
The single-sentence figure includes Python and validation overhead for one call.
The HTTP figure includes response handling and reading JSON bytes, but not parsing or embedding computation.
Compare like workloads before extrapolating. Batching can amortize HTTP costs too.
The 1,200-byte binary size follows from 300 float32 values at four bytes each. JSON size depends on the actual values and formatting.
No service, background listener or container remains after this notebook finishes.

In [ ]:
scratch.cleanup()